# FD001 Baseline Machine Learning for Remaining Useful Life Prediction

This notebook consumes the canonical FD001 file created by notebook 01 and compares Linear Regression, Random Forest, and Gradient Boosting under the shared experiment contract.

**Scope caveat.** FD001 contains 100 training and 100 test trajectories under one sea-level operating condition and one HPC-degradation fault mode. The baseline results apply to this controlled subset and must not be generalized to multi-condition or fan-degradation scenarios without evaluation on FD002–FD004.

**Objectives**

- Load repository-managed training, test, and official RUL files.
- Remove the six FD001 constant sensors and cap the main RUL target at 125.
- Split complete engines between training and validation to prevent leakage.
- Train Linear Regression, Random Forest, and Gradient Boosting baselines.
- Evaluate capped RUL with RMSE, MAE, and R².
- Save comparison, prediction, split, and fitted-model artifacts inside the repository.

Shared decisions are documented in `docs/FD001_EXPERIMENT_CONTRACT.md`. XGBoost tuning is handled in notebook 03, and sequential LSTM/GRU modeling is handled in notebook 04.

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RUL_CAP = 125

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## 1. Load and Validate the Canonical Training Data

Notebook 02 does not download or recreate notebook 01's output. It loads the canonical processed CSV from the repository and fails clearly if notebook 01 has not been run successfully.

In [ ]:
def find_repo_root():
    # Locate the cloned repository from its root or notebooks/ directory.
    starts = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/intelligent-predictive-maintenance"),
    ]
    for start in starts:
        for candidate in [start, *start.parents]:
            if (
                (candidate / "README.md").exists()
                and (candidate / "datasets").exists()
            ):
                return candidate.resolve()

    raise FileNotFoundError(
        "Repository root not found. Run this notebook from the cloned "
        "intelligent-predictive-maintenance repository."
    )


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "datasets" / "raw" / "CMAPSSData"
PROCESSED_PATH = (
    REPO_ROOT / "datasets" / "processed" / "fd001_train_with_rul.csv"
)
TEST_PATH = RAW_DIR / "test_FD001.txt"
TEST_RUL_PATH = RAW_DIR / "RUL_FD001.txt"

required_paths = [PROCESSED_PATH, TEST_PATH, TEST_RUL_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(f"Required project files are missing:\n{missing_text}")

column_names = (
    ["engine_id", "cycle", "setting1", "setting2", "setting3"]
    + [f"sensor{i}" for i in range(1, 22)]
)
expected_columns = column_names + ["max_cycle", "RUL"]

train_fd001 = pd.read_csv(PROCESSED_PATH)
assert train_fd001.columns.tolist() == expected_columns
assert len(train_fd001) == 20631
assert train_fd001["engine_id"].nunique() == 100
assert not train_fd001.duplicated(["engine_id", "cycle"]).any()
assert not train_fd001.isna().any().any()

print(f"Repository root: {REPO_ROOT}")
print(f"Training shape: {train_fd001.shape}")
train_fd001.head()

## 2. Load the Official Test Set and True RUL Labels

The FD001 test trajectories are truncated at some point before failure. The true RUL for the **last** recorded cycle of each test engine is provided separately in `RUL_FD001.txt`. This mirrors how the benchmark is evaluated: given a partial trajectory, predict the remaining life at its final observed cycle.

In [ ]:
test_fd001 = pd.read_csv(
    TEST_PATH,
    sep=r"\s+",
    header=None,
    names=column_names,
)
rul_fd001 = pd.read_csv(
    TEST_RUL_PATH,
    sep=r"\s+",
    header=None,
    names=["RUL"],
)
rul_fd001["engine_id"] = np.arange(1, len(rul_fd001) + 1)

assert test_fd001["engine_id"].nunique() == 100
assert len(rul_fd001) == 100

print(f"Test trajectory shape: {test_fd001.shape}")
print(f"Official RUL-label shape: {rul_fd001.shape}")
rul_fd001.head()

## 3. Baseline Feature Preparation

Notebook 01 identified six sensors with no variation in FD001. Removing them is a basic feature-selection step. The baseline deliberately avoids rolling statistics and other advanced temporal features so notebook 03 can test whether tuning or later feature engineering improves upon a transparent reference model.

In [ ]:
constant_sensors = [
    "sensor1",
    "sensor5",
    "sensor10",
    "sensor16",
    "sensor18",
    "sensor19",
]
setting_columns = ["setting1", "setting2", "setting3"]
sensor_columns = [
    column
    for column in column_names
    if column.startswith("sensor") and column not in constant_sensors
]
feature_columns = setting_columns + sensor_columns

assert len(feature_columns) == 18
print(f"Using {len(feature_columns)} predictors:")
print(feature_columns)

## 4. Capped RUL Target

The main comparison caps RUL at 125 cycles, following the piecewise-linear assumption used in many C-MAPSS experiments. Every primary result in this notebook is explicitly a **capped-RUL** metric. Capped and uncapped results answer different questions and must not be mixed in one comparison.

In [ ]:
train_fd001 = train_fd001.copy()
train_fd001["RUL_capped"] = train_fd001["RUL"].clip(upper=RUL_CAP)

plt.figure(figsize=(7, 4))
plt.hist(train_fd001["RUL"], bins=50, alpha=0.5, label="Raw RUL")
plt.hist(
    train_fd001["RUL_capped"],
    bins=50,
    alpha=0.5,
    label=f"RUL capped at {RUL_CAP}",
)
plt.legend()
plt.title("Effect of the FD001 RUL Cap")
plt.xlabel("Remaining Useful Life (cycles)")
plt.ylabel("Engine-cycle observations")
plt.tight_layout()
plt.show()

## 5. Train / Validation Split (by Engine)

Rows from the same engine are highly correlated across consecutive cycles. Splitting randomly by row would leak information between train and validation sets, since near-identical cycles from the same engine could land on both sides. Instead, the split is done at the **engine level**: entire engines go to either the training set or the validation set.

In [ ]:
engine_ids = np.sort(train_fd001["engine_id"].unique())
train_ids, validation_ids = train_test_split(
    engine_ids,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_split = train_fd001[train_fd001["engine_id"].isin(train_ids)].copy()
validation_split = train_fd001[
    train_fd001["engine_id"].isin(validation_ids)
].copy()

X_train = train_split[feature_columns]
y_train = train_split["RUL_capped"]
X_validation = validation_split[feature_columns]
y_validation = validation_split["RUL_capped"]

assert set(train_ids).isdisjoint(validation_ids)
assert len(train_ids) == 80
assert len(validation_ids) == 20

print(f"Training engines: {len(train_ids)}; rows: {len(X_train):,}")
print(
    f"Validation engines: {len(validation_ids)}; "
    f"rows: {len(X_validation):,}"
)

## 6. Prepare the Official Test Set (Last Cycle per Engine)

For each test engine, only the final observed cycle is scored, paired with the true RUL from `RUL_FD001.txt`.

In [ ]:
test_last_cycle = (
    test_fd001.sort_values(["engine_id", "cycle"])
    .groupby("engine_id", as_index=False)
    .tail(1)
    .reset_index(drop=True)
    .merge(rul_fd001, on="engine_id", how="left", validate="one_to_one")
)
test_last_cycle["RUL_capped"] = test_last_cycle["RUL"].clip(
    upper=RUL_CAP
)

X_test = test_last_cycle[feature_columns]
y_test = test_last_cycle["RUL_capped"]
y_test_uncapped = test_last_cycle["RUL"]

assert len(X_test) == 100
assert not test_last_cycle["RUL"].isna().any()

print(f"Official test engines: {len(X_test)}")
test_last_cycle[["engine_id", "cycle", "RUL", "RUL_capped"]].head()

## 7. Feature Scaling

The scaler is fit on training engines only. Linear Regression benefits from scaling. The tree models do not require it, but Mina's original baseline applied the same transformed matrix to all three models; that choice is retained so the revised results remain directly comparable with the original notebook.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

## 8. Baseline Models

### 8.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)


### 8.2 Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_scaled, y_train)


### 8.3 Gradient Boosting Regressor

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
)
gb.fit(X_train_scaled, y_train)


## 9. Capped-RUL Model Evaluation and Comparison

Each model is evaluated on the held-out validation engines and the official FD001 test endpoints using capped RUL at 125 cycles. The official test results describe final generalization and are not used to select baseline hyperparameters.

In [ ]:
def evaluate_model(model, features, target):
    predictions = model.predict(features)
    metrics = {
        "RMSE": mean_squared_error(target, predictions) ** 0.5,
        "MAE": mean_absolute_error(target, predictions),
        "R2": r2_score(target, predictions),
    }
    return metrics, predictions


models = {
    "Linear Regression": lr,
    "Random Forest": rf,
    "Gradient Boosting": gb,
}

results = []
test_predictions = {}

for model_name, model in models.items():
    validation_metrics, _ = evaluate_model(
        model,
        X_validation_scaled,
        y_validation,
    )
    test_metrics, model_test_predictions = evaluate_model(
        model,
        X_test_scaled,
        y_test,
    )
    test_predictions[model_name] = model_test_predictions
    results.append(
        {
            "Model": model_name,
            "Target": f"RUL capped at {RUL_CAP}",
            "Validation_RMSE": validation_metrics["RMSE"],
            "Validation_MAE": validation_metrics["MAE"],
            "Validation_R2": validation_metrics["R2"],
            "Test_RMSE": test_metrics["RMSE"],
            "Test_MAE": test_metrics["MAE"],
            "Test_R2": test_metrics["R2"],
        }
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("Test_RMSE")
    .reset_index(drop=True)
)
results_df.round(3)

In [ ]:
metrics_to_plot = [
    "Validation_RMSE",
    "Test_RMSE",
    "Validation_MAE",
    "Test_MAE",
]

fig, axis = plt.subplots(figsize=(10, 5))
results_df.set_index("Model")[metrics_to_plot].plot(kind="bar", ax=axis)
axis.set_title("Baseline Comparison — RUL Capped at 125 Cycles")
axis.set_ylabel("Error (cycles)")
axis.set_xticklabels(results_df["Model"], rotation=0)
plt.tight_layout()
plt.show()

## 10. Feature Importance (Tree-Based Models)

Inspects which settings and sensors the Random Forest and Gradient Boosting models rely on most, and checks whether this lines up with the sensors that showed the strongest correlation with RUL in the notebook 01 EDA (`sensor11`, `sensor4`, `sensor12`, `sensor7`, `sensor15`).

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "rf_importance": rf.feature_importances_,
    "gb_importance": gb.feature_importances_,
}).sort_values("rf_importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(importance_df["feature"], importance_df["rf_importance"])
axes[0].invert_yaxis()
axes[0].set_title("Random Forest Feature Importance")

order = importance_df.sort_values("gb_importance", ascending=False)
axes[1].barh(order["feature"], order["gb_importance"])
axes[1].invert_yaxis()
axes[1].set_title("Gradient Boosting Feature Importance")

plt.tight_layout()
plt.show()

importance_df


## 11. Predicted vs. Actual RUL on the Test Set

Visualizes how closely each model's predictions track the true RUL for the official test engines. Points near the diagonal line indicate accurate predictions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)

for ax, (name, preds) in zip(axes, test_predictions.items()):
    ax.scatter(y_test, preds, alpha=0.6, edgecolor="k", linewidth=0.3)
    lims = [0, max(y_test.max(), preds.max()) + 5]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("Actual RUL")
    ax.set_ylabel("Predicted RUL")

plt.tight_layout()
plt.show()


## 12. Save Repository-Managed Baseline Artifacts

Small tables and predictions are saved under `reports/`; reusable fitted objects are saved under `models/`. The exact engine split is also saved so notebook 03 and team reviewers can verify an identical comparison.

In [ ]:
reports_dir = REPO_ROOT / "reports"
models_dir = REPO_ROOT / "models"
reports_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

comparison_path = reports_dir / "baseline_model_comparison.csv"
predictions_path = reports_dir / "baseline_test_predictions.csv"
split_path = reports_dir / "baseline_engine_split.csv"

results_df.to_csv(comparison_path, index=False)

prediction_output = test_last_cycle[
    ["engine_id", "cycle", "RUL", "RUL_capped"]
].copy()
for model_name, predictions in test_predictions.items():
    safe_name = model_name.lower().replace(" ", "_")
    prediction_output[f"prediction_{safe_name}"] = predictions
prediction_output.to_csv(predictions_path, index=False)

pd.DataFrame(
    {
        "engine_id": np.concatenate([train_ids, validation_ids]),
        "partition": (
            ["train"] * len(train_ids)
            + ["validation"] * len(validation_ids)
        ),
    }
).to_csv(split_path, index=False)

joblib.dump(scaler, models_dir / "baseline_scaler.pkl")
joblib.dump(lr, models_dir / "baseline_linear_regression.pkl")
joblib.dump(rf, models_dir / "baseline_random_forest.pkl")
joblib.dump(gb, models_dir / "baseline_gradient_boosting.pkl")

print("Saved repository artifacts:")
for path in [comparison_path, predictions_path, split_path]:
    print(f"- {path.relative_to(REPO_ROOT)}")

## Baseline Summary and Limitations

This notebook establishes reproducible FD001 baseline regressors using the shared engine-level split, 18-feature set, capped-RUL target, and official endpoint evaluation. Notebook 03 can now reproduce or cross-check these results using saved comparison and split files.

The results are limited to one simulated operating condition and one HPC-degradation fault mode. They do not demonstrate performance under the six operating conditions in FD002/FD004 or the fan-degradation mode in FD003/FD004. Hyperparameters are fixed baseline settings rather than tuned optimal values; tuning is intentionally reserved for Ivan's advanced-ML component.